## Transform Drivers Data
- Read bronze driver table
- Keep only the columns required for analytics(Drop url column)
- Standardise column name using snakecase(driverId->driver_id)
- Rename columns to make them more meaningful(name->constructor_name)
- Remove duplicate records.
- Transform values of column nationality to Title Case
- Write the transformed data to silver driver table

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table=f"{catalog_name}.{bronze_schema}.drivers"
silver_table=f"{catalog_name}.{silver_schema}.drivers"

In [0]:
import pyspark.sql.functions as F

### Step 1 Read bronze Drivers table

In [0]:
drivers_df=spark.table(bronze_table)


### Srep 2 -Keep only the columns required for analytics(Drop url column)

In [0]:
#flexible   `

drivers_selected_df=(
    drivers_df.drop("url")
)

### Step 3 - Stamdardise column name
- Standardise column name using snakecase(driverId->driver_id)


In [0]:
drivers_renamed_df=(
    drivers_selected_df.withColumnRenamed("driverId","driver_id")
                       .withColumnRenamed("dateOfBirth","date_of_birth")
                )
#display(drivers_renamed_df)

### Step 4 Concatenate name.givenName and name.familyName to Create a new column called as driver_name

In [0]:
drivers_concatenated_df=(
    drivers_renamed_df
    .withColumn("driver_name",
                F.initcap(F.concat_ws(" ",F.col("name.givenName"),F.col("name.familyName"))))
    .drop("name")
    
)

#dropduplicates 

In [0]:
display(drivers_concatenated_df)

driver_id,date_of_birth,nationality,ingestion_timestamp,source_file,driver_name
abate,1932-07-10,italian,2026-10-06T17:18:27.491059Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Carlo Abate
abecassis,1913-03-21,british,2026-10-06T17:18:27.491059Z,dbfs:/Volumes/formula1/landing/files/drivers.json,George Abecassis
acheson,1957-11-27,british,2026-10-06T17:18:27.491059Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Kenny Acheson
adams,1969-11-19,belgian,2026-10-06T17:18:27.491059Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Philippe Adams
ader,1913-12-15,american,2026-10-06T17:18:27.491059Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Walt Ader
adolff,1921-11-05,german,2026-10-06T17:18:27.491059Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Kurt Adolff
agabashian,1913-08-21,american,2026-10-06T17:18:27.491059Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Fred Agabashian
ahrens,1940-04-19,german,2026-10-06T17:18:27.491059Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Kurt Ahrens
aitken,1995-09-23,british,2026-10-06T17:18:27.491059Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Jack Aitken
albers,1979-04-16,dutch,2026-10-06T17:18:27.491059Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Christijan Albers


### Step -5 Remove duplicates

In [0]:
#dropduplicates - you can give column name and distinct will be applied on that particular column only
drivers_distinct_df=drivers_concatenated_df.dropDuplicates(["driver_id"])
#display(drivers_distinct_df)


### Step -6 Transform values of column nationality to Title Case

In [0]:
drivers_final_df=(
    drivers_distinct_df
    .withColumn("nationality",F.initcap(F.col("nationality")))
)
#display(drivers_final_df)

### Step-7 Write the transformed data to silver drivers table

In [0]:
(
   drivers_final_df
    .write
    .mode("overwrite")
    .format("delta")
    .option('overwriteSchema', 'true')
    .saveAsTable(silver_table)
)

In [0]:
display(spark.read.table(silver_table))

driver_id,date_of_birth,nationality,ingestion_timestamp,source_file,driver_name
Cannoc,1933-06-21,Canadian,2026-10-06T17:18:27.491059Z,dbfs:/Volumes/formula1/landing/files/drivers.json,John Cannon
Changy,1922-02-05,Belgian,2026-10-06T17:18:27.491059Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Alain De Changy
abate,1932-07-10,Italian,2026-10-06T17:18:27.491059Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Carlo Abate
abecassis,1913-03-21,British,2026-10-06T17:18:27.491059Z,dbfs:/Volumes/formula1/landing/files/drivers.json,George Abecassis
acheson,1957-11-27,British,2026-10-06T17:18:27.491059Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Kenny Acheson
adamich,1941-10-03,Italian,2026-10-06T17:18:27.491059Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Andrea De Adamich
adams,1969-11-19,Belgian,2026-10-06T17:18:27.491059Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Philippe Adams
ader,1913-12-15,American,2026-10-06T17:18:27.491059Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Walt Ader
adolff,1921-11-05,German,2026-10-06T17:18:27.491059Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Kurt Adolff
agabashian,1913-08-21,American,2026-10-06T17:18:27.491059Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Fred Agabashian
